# 3.3 Object Detection with LLMs

## Object Detection
In the previous lesson, we used a multimodal LLM for image understanding. If we need to approach a specific target, we need to know the target's position -- this is what object detection provides: bounding boxes (bbox).

<img src="img/bbox.png" width='640px' />


### **Object Detection**
**Object Detection** is a computer vision task that identifies objects in images or video and locates their positions (typically represented by bounding boxes). It not only classifies objects ("person", "car", "dog") but also localizes them within the image. Unlike **image classification** (single label) and **image segmentation** (pixel-level), object detection handles multiple objects of multiple classes simultaneously.

---

#### 1. **Traditional Methods** (pre-2010)
- **Pipeline**: Sliding window generation -> Feature extraction (SIFT, HOG) -> Classifier (SVM) classification.
- **Disadvantages**: High computational cost, poor robustness, limited to specific scenes.

#### 2. **Deep Learning Methods** (2012-2020)
- **Pipeline**:
  1. **Feature extraction**: Using CNNs to extract image features
  2. **Bounding box regression**: Predicting positions (anchor-based or anchor-free)
  3. **Classification and filtering**: Using IoU for evaluation, Non-Maximum Suppression (NMS) for filtering

- **Categories**:
  1. **Two-Stage** (region-based): Generate proposals first (Faster R-CNN with RPN), then classify. Higher precision but slower.
  2. **One-Stage** (direct): Predict boxes and classes simultaneously (YOLO). Faster but may struggle with small objects.
  3. **YOLO series**: Real-time, fast, widely used in monitoring scenarios.


#### 3. **Current State-of-the-Art**

<img src="img/s3-1.png" width='640px' />

1. Attention-based, Transformer-Based (DETR): No NMS needed, but training is resource-intensive.
2. Under the LLM paradigm, moving from closed-set (predefined classes) to open-set (open vocabulary) detection.
3. Able to detect targets based on natural language descriptions or examples, enabling zero-shot detection with strong multimodal alignment and generalization.


Open-vocabulary object detection with LLMs:

<img src="img/dino.jpg" width='800px' />

## Resources

- DINO open-source: https://github.com/IDEA-Research/DINO
- Grounding-DINO open-source: https://github.com/IDEA-Research/GroundingDINO
- DINO online playground: https://cloud.deepdataspace.com/playground/grounding_dino
- DINO API docs: https://cloud.deepdataspace.com/docs#model/dinox


<img src='img/dino-s.png' width='720px' />

Grounding DINO: 0.1 yuan/call

Registration includes 20 free credits, approximately 200 calls.

Get your token: https://cloud.deepdataspace.com/dashboard/token-key

In [ ]:
!git clone https://github.com/deepdataspace/dds-cloudapi-sdk.git

In [ ]:
!pip install torch

In [ ]:
!pip install dds_cloudapi_sdk

In [ ]:
# Note: After installation completes, you may need to restart the kernel

In [ ]:
from dds_cloudapi_sdk.tasks.v2_task import create_task_with_local_image_auto_resize
from dds_cloudapi_sdk import Config
from dds_cloudapi_sdk import Client
from dds_cloudapi_sdk.visualization_util import visualize_result

token = "929cfa1acbe0ef4748bcbb1c4b29703d"
config = Config(token)
client = Client(config)

In [ ]:
img_path = "./img/p-4-1-s.png"
output_path = "./img"
text = "yellow duck.Cola"
# 2. Create a task with proper parameters. 
task = create_task_with_local_image_auto_resize(
    api_path="/v2/task/dinox/detection",
    api_body_without_image={
        "model": "DINO-X-1.0",
        # "image": infer_image_url, # not needed for local image
        "prompt": {
            "type": "text",
            "text": text
        },
        "targets": ["bbox"],
        "bbox_threshold": 0.25,
        "iou_threshold": 0.8
    },
    image_path=img_path)

# 3. Run the task.
# task.set_request_timeout(10)  # set the request timeout in seconds，default is 5 seconds
client.run_task(task)
image_pil = visualize_result(image_path=img_path, result=task.result, output_dir=output_path)

In [ ]:
from IPython.display import display # type: ignore
from PIL import Image
img_path = "./img/annotated_image.jpg"
image_pil = Image.open(img_path)
display(image_pil)

## Reading Images with OpenCV

In [ ]:
import cv2 # type: ignore
local_image_path = "./img/p-4-1-s.png"
image = cv2.imread(local_image_path)
rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
import uuid
bgr_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
# Generate random filename (with extension)
file_name = f"random_{uuid.uuid4().hex}.png"  # Example: random_1a2b3c4d5e.png
cv2.imwrite('./img/' + file_name, bgr_image)

In [ ]:
task = create_task_with_local_image_auto_resize(
    api_path="/v2/task/dinox/detection",
    api_body_without_image={
        "model": "DINO-X-1.0",
        # "image": infer_image_url, # not needed for local image
        "prompt": {
            "type": "text",
            "text": "yellow duck.Cola"
        },
        "targets": ["bbox"],
        "bbox_threshold": 0.25,
        "iou_threshold": 0.8
    },
    image_path="./img/"+file_name)

# 3. Run the task.
# task.set_request_timeout(10)  # set the request timeout in seconds，default is 5 seconds
client.run_task(task)

In [ ]:
results=task.result
results